In [ ]:
## For Google Colaboratory 

from google.colab import drive
drive.mount('/content/drive/',force_remount=True)
%cd /content/drive/MyDrive/Pesquisa/2024_glaucoma/

# Dataset

In [ ]:
## Setting the random seed and enabling determinism
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
from torch.utils.data import Dataset
from torchvision.io import decode_image
from PIL import Image
from torchvision.transforms import ToTensor

## Custom dataset class creation
## Loads images on the fly, respecting the directory structure
class DatasetGlaucoma(Dataset):
    def __init__(self, img_dir, transform=None, target_transform=None):
        self.img_dir = img_dir
        imgs_total = []
        self.labels = []

        self.count_nrg = 0
        self.count_rg = 0

        for i in os.listdir(img_dir + "NRG/"):
            imgs_total.append((os.path.join(img_dir + "NRG/", i), torch.tensor(0)))
            self.labels.append(0)
            self.count_nrg += 1
        for i in os.listdir(img_dir + "RG/"):
            imgs_total.append((os.path.join(img_dir + "RG/", i), torch.tensor(1)))
            self.labels.append(1)
            self.count_rg += 1

        self.imgs_total = imgs_total
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.imgs_total)

    def len_division(self):
        return {"NRG": self.count_nrg, "RG": self.count_rg}

    def __getitem__(self, idx):
        (image_dir, label) = self.imgs_total[idx]
        with Image.open(image_dir) as img:
            image = img.convert('RGB')
        if self.transform:
                image = self.transform(image)
        return image, label

In [ ]:
from torch.utils.data import Dataset
from torchvision.io import decode_image
from PIL import Image
from torchvision.transforms import ToTensor

## Custom dataset class creation
## Loads images on the fly, respecting the directory structure
class DatasetGlaucomaTVT(Dataset):
    def __init__(self, img_dir, transform=None, target_transform=None):
        self.img_dir = img_dir

        self.imgs_train = DatasetGlaucoma(img_dir + "train/", transform, target_transform)
        self.imgs_val = DatasetGlaucoma(img_dir + "val/", transform, target_transform)
        self.imgs_test = DatasetGlaucoma(img_dir + "test/", transform, target_transform)

    def __len__(self):
        return -1

    def __getitem__(self, idx):
        return -1

    def get_dataset(self, type):
        if (type == "train"):
            return self.imgs_train
        if (type == "val"):
            return self.imgs_val
        if (type == "test"):
            return self.imgs_test
        return -1

In [ ]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import os
import torch
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import torch.backends.cudnn as cudnn

cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

## Sets the random seed to 123 for reproducibility
SEED = 123
generator = torch.Generator().manual_seed(SEED)
seed_everything(SEED)

## Defines the path to the base dataset
datasetDirectory = "./datasets/discos_15371_comAG2/"
#datasetDirectory = "./datasets/discos_11732_semAG/"

## Instantiates the complete dataset object
dataset_nome = datasetDirectory[datasetDirectory.rindex("/", 0, (len(datasetDirectory) - 1)) + 1:len(datasetDirectory) - 1]

In [ ]:
## For Local Machine

ssd_datasetDirectory = "/home/jovyan/SSD/" + datasetDirectory[11:]
if (os.path.exists(ssd_datasetDirectory)):
    full_dataset = DatasetGlaucomaTVT(
        img_dir=ssd_datasetDirectory,
        transform=ToTensor()
    )
else: 
    full_dataset = DatasetGlaucomaTVT(
        img_dir=datasetDirectory,
        transform=ToTensor()
    )

In [ ]:
## For Google Colaboratory 

google_datasetDirectory = "/content/" + datasetDirectory[11:]
full_dataset = DatasetGlaucomaTVT(
    img_dir=google_datasetDirectory,
    transform=ToTensor()
)

# Model (Backbone + MLP)

## Basic Functions

In [ ]:
from google.oauth2.service_account import Credentials
import gspread as gs
import gspread_dataframe as gs_d
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

## Add a .env with a GOOGLE_SHEET_ID variable
google_sheet_id = os.getenv('GOOGLE_SHEET_ID')

## Functions created for automating Google Spreadsheet data annotation
## Helper function to authenticate and connect to a Google Spreadsheet using service account credentials
def connnect_drive(sheet_id, worksheet):
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets"
    ]

    creds = Credentials.from_service_account_file('./credentials.json', scopes=scopes)

    client = gs.authorize(creds)

    return client.open_by_key(sheet_id)

## Saves model metadata and configurations to a specified sheet
def saveCSV_RAY(name, metadata, sheet_id, worksheet=None):
    sheet = connnect_drive(sheet_id, worksheet)

    if (worksheet != None):
        sheet = sheet.worksheet(worksheet)
    else:
        sheet = sheet.sheet1

    data = {"nome": name}

    ## Flattens nested dictionaries into a single dictionary
    dict = {}
    for i in metadata.keys():
        for key, value in metadata[i].items():
            if isinstance(value, (list)) and len(value) == 0:
                metadata[i][key] = None
            if (type(value) == list) or (type(value) == dict):
                metadata[i][key] = str(metadata[i][key])
        dict = (dict | metadata[i])

    ## Structures and orders the columns: 'nome' goes first, followed by metadata
    lista = list((dict | data).keys())
    lista.remove("nome")
    lista.insert(0, "nome")
    df_novo = pd.DataFrame((data | dict), index=[0])
    df_novo = df_novo.loc[:, lista]

    df_carregado = gs_d.get_as_dataframe(sheet, index_col=0)
    if (len(df_carregado) != 0):
        df = pd.concat([df_carregado, df_novo], ignore_index=True)
        del df_carregado
    else:
        df = df_novo

    gs_d.set_with_dataframe(sheet, df, include_index=True)
    del df, df_novo

## Retrieves saved model metrics from a Google Sheet based on the model's metadata
def restoreModel_CSV(name, metadata, sheet_id, worksheet=None):
    sheet = connnect_drive(sheet_id, worksheet)

    if (worksheet != None):
        sheet = sheet.worksheet(worksheet)
    else:
        sheet = sheet.sheet1

    df_carregado = gs_d.get_as_dataframe(sheet, index_col=1)

    try:
        model_data = df_carregado.loc[name]
    except KeyError:
        return -1

    ## Filters entries with identical names to find the exact metadata match
    temp_model_data = 0
    achou = True

    lista = list(df_carregado.keys())

    for entry in ["epoch", "net_state_dict", "optimizer_state_dict", "train_loss", "test_loss"]:
        lista.remove(entry)

    ## Iterates through rows to find a strict match for all metadata parameters
    if (len(model_data) < 10):
        for i in range(0, len(model_data)):
            achou = True
            temp_model_data = (model_data.iloc[i])[lista]
            for (key, value) in metadata.items():
                if (temp_model_data[key] != value):
                    achou = False
                    break

            if (achou == True):
                model_data = model_data.iloc[i]
                break

    if (achou != True):
        return -1

    for (key, value) in metadata.items():
        if (model_data[key] != value):
            return -1

    checkpoint_data = {
        "train_loss": model_data["train_loss"],
        "test_loss": model_data["test_loss"],
    }

    return checkpoint_data

In [ ]:
import pickle
import os
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
from sympy import true
from tqdm.notebook import tqdm
import json
import numpy as np
import pandas as pd
import time
from sklearn.metrics import roc_auc_score, confusion_matrix
import torchvision.transforms.functional as TF

formatted_time = time.strftime("%d-%m-%y_%H-%M-%S", time.localtime(time.time()))

## Helpers functions for training
## Model checkpointing system: a workaround for Google Colab session timeouts and limits
def backupAndRestore(model, path, atualEpoch, batchSize):
    path_dir = "./modelos_treinados/backup_part/Pytorch/"
    os.makedirs(path_dir, exist_ok=True)

    path = path_dir + path

    if os.path.exists(path):
        with open(f'{path}/training_metadata.json', "r") as f:
            data = json.load(f)

        if (data["epoch"] > atualEpoch) and (data["batch"] == batchSize):
            model.load_state_dict(torch.load(f'{path}/latest.weights.pth', weights_only=True))
            return (model, data["epoch"])

        if (data["batch"] != batchSize):
            raise Exception("O batchSize treinado é diferente do modelo salvo na pasta. Impossivel salvar o modelo")
    else:
        os.makedirs(path, exist_ok=True)

    torch.save(model.state_dict(), f'{path}/latest.weights.pth')

    data = {"epoch": atualEpoch, "batch": batchSize}

    with open(f'{path}/training_metadata.json', "w") as f:
        json.dump(data, f)

    return (model, atualEpoch)

def calc_roc(model, error, test_ds):
    model.eval()
    dataloader = test_ds

    running_acc = 0
    running_loss = 0
    y_pred = []
    y_true = []

    with torch.set_grad_enabled(False):
        for i, (images, labels) in enumerate(tqdm(dataloader)):
            batch_acc = 0
            batch_loss = 0

            for pos_batch in range(0, len(images)):
                img = images[pos_batch].to(device)
                label = labels[pos_batch].to(device).float().unsqueeze_(0)

                output = model(img)
                loss = error(output, label)
                loss = loss / len(images)
                batch_loss += loss.item()


                output = torch.sigmoid(output)
                output = (output > custom_threshold).float()
                batch_acc += (output == label.data).float().mean().item()

                y_pred.append(output.cpu())
                y_true.append(label.cpu())

            running_loss += batch_loss
            running_acc += batch_acc / len(images)

    y_pred = torch.cat(y_pred).numpy().ravel()
    y_true   = torch.cat(y_true).numpy().ravel()

    epoch_acc = running_acc / len(dataloader)
    epoch_loss = running_loss / len(dataloader)

    return (y_pred, y_true)

def calc_statisticsV2_pytorch(model, error, test_ds, custom_threshold=0.5, enable_tta=False, visible=True):
    """
    Calculates the performance statistics for the specified model.
    :param model: (NeuralNetworkCustom) The model used for evaluate.
    :param error: (torch.nn.modules.loss) The PyTorch loss function used to compute the error.
    :param test_ds: (DataLoader) The DataLoader providing the desire dataset.
    :return: None. Prints the evaluation metrics to the console.
    """

    tp = 0 # true positive
    tn = 0 # true negative
    fp = 0 # false positive
    fn = 0 # false negative

    model.eval()
    dataloader = test_ds

    running_loss = 0.0
    total_samples = 0

    ## Lists for vectorized accumulation of predictions and labels
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", disable=not(visible)):
            images = images.to(device)
            labels = labels.to(device).float().view(-1, 1)

            ###############################################
            ## TTA
            if (enable_tta == False):
                outputs = model(images)
            else:
                logits_orig = model(images)

                img_hflip = TF.hflip(images)
                logits_hflip = model(img_hflip)

                img_vflip = TF.vflip(images)
                logits_vflip = model(img_vflip)

                outputs = (logits_orig + logits_hflip + logits_vflip) / 3.0
            ###############################################

            loss = error(outputs, labels)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            probs = torch.sigmoid(outputs)
            preds = (probs > custom_threshold).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()

    epoch_loss = running_loss / total_samples

    # Recall (Sensitivity) or True Positive Rate (TPR)
    # TPR = TP / (TP + FN)
    try:
        tpr = tp / (tp + fn)
    except ZeroDivisionError:
        print("ZeroDivisionError")
        tpr = 0

    # Precision or Positive Predictive Value (PPV)
    # PPV = TP / (TP + FP)
    try:
        ppv = tp / (tp + fp)
    except ZeroDivisionError:
        print("ZeroDivisionError")
        ppv = 0

    # Specificity or Selectivity or True Negative Rate (TNR)
    try:
        tnr = tn / (tn + fp)
    except ZeroDivisionError:
        print("ZeroDivisionError")
        tnr = 0

    # Accuracy (ACC)
    try:
        acc = (tp + tn) / (tp + tn + fp + fn)
    except ZeroDivisionError:
        print("ZeroDivisionError")
        acc = 0

    # F1 score
    try:
        f1 = 2 * (ppv * tpr) / (ppv + tpr)
    except ZeroDivisionError:
        print("ZeroDivisionError")
        f1 = 0

    if (visible):
        print('True Positive (TP): {}'.format(tp))
        print('True Negative (TN): {}'.format(tn))
        print('False Positive (FP): {}'.format(fp))
        print('False Negative (FN): {}'.format(fn))
        print("Recall (TPR): {:.4f}".format(tpr))
        print('Precisão (PPV): {:.4f}'.format(ppv))
        print('Especificidade (TNR): {:.4f}'.format(tnr))
        print('Acurácia (ACC): {:.4f}'.format(acc))
        print('F1 score (F1): {:.4f}'.format(f1))

    try:
        pAUC = roc_auc_score(all_labels, all_probs, max_fpr=0.1)
    except ValueError:
        print("Aviso: Apenas uma classe presente nos labels. pAUC definido como 0.")
        pAUC = 0.0

    return (tp, tn, fp, fn, acc, tpr, ppv, tnr, f1, pAUC)

def plotGraphics(path, list_epoch, acc_train, acc_val, loss_train, loss_val):
    for (phrase, list_train, list_val) in [("Accuracy", acc_train, acc_val), ("Loss", loss_train, loss_val)]:
        if (loss_val == []):
            break
        ax = plt.figure().gca()
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        plt.plot(list_epoch, list_train, label="Train", linewidth=2.0)
        plt.plot(list_epoch, list_val, label="Validation", linewidth=2.0)
        plt.title(phrase)
        plt.xlabel("Epoch")
        plt.ylabel(phrase)
        plt.legend()
        plt.savefig(f'{path}plt_{phrase}.png')
        #plt.show(block=True)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

## Save model weights, results and graphs. (Exclusive for test dataset)
def saveModel(obj_model, model_name, metadata, list_epoch, acc_train, acc_val, loss_train, loss_val, test_ds, error, model_path="./modelos_treinados/Pytorch/", nome_projeto=None):
    (tp, tn, fp, fn, acc, recall, prec, espec, f1) = calc_statisticsV2_pytorch(obj_model, error, test_ds)

    nome_pasta = ""

    data = {"config_modelo":
                metadata,
            "metricas":
                {"acc_train": acc_train,
                 "acc_val": acc_val,
                 "loss_train": loss_train,
                 "loss_val": loss_val},
            "metricas_datasetTeste":
                {"tp": tp,
                 "tn": tn,
                 "fp": fp,
                 "fn": fn,
                 "acurácia": acc,
                 "recall": recall,
                 "precisão": prec,
                 "especificação": espec,
                 "f1-score": f1}}

    saveCSV_RAY(model_name, data, google_sheet_id, google_sheet_name)

    model_path = f'{model_path}{model_name}/'
    os.makedirs(model_path, exist_ok=True)

    plotGraphics(model_path, list_epoch, acc_train, acc_val, loss_train, loss_val)

    ## Create a folder with model weights, results and graphs.
    with open(f'{model_path}training_metadata.json', "w") as file:
        json.dump(data, file, cls=NpEncoder) ## Para guardar algumas infos importantes do modelo

    with open(f'{model_path}model_class.pkl', 'wb') as file:
        pickle.dump(obj_model, file) ## Para guardar a classe do Modelo, pois pode ser que tenha alguma classe com versões diferentes

    torch.save(obj_model.state_dict(), f'{model_path}{model_name}.pth') ## Para guardar o dicionário do modelo, o que foi treinado
    print(f'Saved on: {model_path}')

    del obj_model

from pathlib import Path
import os
import shutil

def get_notebook_path():
    try:
        import ipynbname
        return ipynbname.path()
    except Exception:
        pass

    try:
        from google.colab import _message
        resp = _message.blocking_request("get_ipynb")
        name = resp["ipynb"]["metadata"]["colab"]["name"]
        return Path(name)
    except Exception:
        pass

    try:
        from IPython import get_ipython
        ip = get_ipython()
        session = ip.user_ns.get("__session__")
        if session:
            return Path(session)
    except Exception:
        pass

    return None

def save_notebook_snapshot(run_dir, notebook_path=None, must_exist=True):
    run_dir = Path(run_dir)

    nb = notebook_path or get_notebook_path()
    if nb is None:
        raise RuntimeError("Não consegui detectar o notebook atual.")

    nb = Path(nb)

    if must_exist and not nb.exists():
        raise FileNotFoundError(
            f"Notebook detectado, mas não encontrado no disco: {nb}\n"
            "Salve o notebook manualmente com Ctrl+S e tente de novo."
        )

    dst = run_dir / nb.name
    shutil.copy2(nb, dst)
    return dst

## Save model weights, results and graphs. (For test and validation dataset)
def saveModel_VTD(obj_model, model_name, enable_tta, metadata, list_epoch, acc_train, acc_train_val, loss_train, loss_val, val_ds, test_ds, error, model_path="./modelos_treinados/Pytorch/", nome_projeto=None):
    print("Test dataset")
    (tp_test, tn_test, fp_test, fn_test, acc_test, recall_test, prec_test, espec_test, f1_test, pAUC_test) = calc_statisticsV2_pytorch(obj_model, error, test_ds, enable_tta=enable_tta)

    nome_pasta = ""

    data = {"config_modelo":
            metadata,
        "metricas":
            {"acc_train": acc_train,
             "loss_train": loss_train},
        "metricas_datasetTeste":
            {"tp_teste": tp_test,
             "tn_teste": tn_test,
             "fp_teste": fp_test,
             "fn_teste": fn_test,
             "acurácia_teste": acc_test,
             "recall_teste": recall_test,
             "precisão_teste": prec_test,
             "especificação_teste": espec_test,
             "f1-score_teste": f1_test,
             "pAUC_teste": pAUC_test},}

    ## Replace the placeholders with your own Google Sheet ID and Name
    saveCSV_RAY(model_name, data, google_sheet_id, google_sheet_name)

    model_path = f'{model_path}{model_name}/'
    os.makedirs(model_path, exist_ok=True)

    plotGraphics(model_path, list_epoch, acc_train=acc_train, acc_val=acc_train_val, loss_train=loss_train, loss_val=loss_val)

    with open(f'{model_path}training_metadata.json', "w") as file:
        ## Stores essential model metadata
        json.dump(data, file, cls=NpEncoder)

    ## Saves the model's state_dict (the trained weights)
    torch.save(obj_model.state_dict(), f'{model_path}{model_name}.pth')
    
    ## If error, continues
    try:
        saved_nb = save_notebook_snapshot(model_path)
    except Exception as e:
        print(f"[warn] não consegui salvar o notebook: {e}")
        
    print(f'Saved on: {model_path}')

    del obj_model

In [ ]:
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
import numpy as np
import copy

def threshold_optimization(model, val_ds, batch_size, error):
    custom_threshold = 0.5
    
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    
    model.eval()
    dataloader = val_ds
    
    running_loss = 0.0
    total_samples = 0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Threshold Optimization: Evaluating "):
            images = images.to(device)
            labels = labels.to(device).float().view(-1, 1)
    
            outputs = model(images)
            loss = error(outputs, labels)
    
            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size
    
            probs = torch.sigmoid(outputs)
            preds = (probs > custom_threshold).float()
    
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())
    
        all_preds = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        all_probs = torch.cat(all_probs).numpy()
    
    thresholds_to_test = np.linspace(0.1, 0.9, 81)
    best_acc = 0.0
    best_threshold_acc = 0.5
    
    ## Exhaustive search for the highest possible accuracy
    for t in thresholds_to_test:
        preds = (all_probs >= t).astype(int)
        current_acc = accuracy_score(all_labels, preds)
    
    
        if current_acc > best_acc:
            best_acc = current_acc
            best_threshold_acc = t
    
    print(f"O Threshold Ótimo focado apenas em ACURÁCIA é: {best_threshold_acc:.4f} (Acc Esperada: {best_acc:.4f})")
    return best_threshold_acc

def tta_val(model, test_ds, batch_size, error, best_threshold_acc, visible=True):
    custom_threshold = 0.5
    
    experimentos = [
        {"name": "Normal",          "threshold": 0.50,              "enable_tta": False},
        {"name": "Normal+TTA",      "threshold": 0.50,              "enable_tta": True},
        {"name": "Threshold",       "threshold": best_threshold_acc, "enable_tta": False},
        {"name": "Threshold+TTA",   "threshold": best_threshold_acc, "enable_tta": True},
    ]
    
    def to_scalar(x):
        if hasattr(x, "item"):
            return x.item()
        return x
    
    metric_names = {
        0: "tp",
        1: "tn",
        2: "fp",
        3: "fn",
        4: "acc",
        5: "recall",
        6: "precision",
        7: "specificity",
        8: "f1",
        9: "pAUC",
    }
        
    rows = []
    
    for exp in experimentos:
        stats = calc_statisticsV2_pytorch(
            model,
            error,
            test_ds,
            exp["threshold"],
            enable_tta=exp["enable_tta"],
            visible=visible
        )
    
        row = {
            "name": exp["name"],
            "threshold": exp["threshold"],
            "tta": exp["enable_tta"],
        }
    
        for i, value in enumerate(stats):
            col = metric_names.get(i, f"metric_{i}")
            row[col] = to_scalar(value)
    
        rows.append(row)
    
    df_metricas = pd.DataFrame(rows).set_index("name")
    df_metricas = df_metricas.round(4)
    
    return df_metricas

In [ ]:
## Save model weights, results and graphs with TTA and Threshold Optimization built-in. (For test and validation dataset)
def saveModelV2(obj_model, model_name, enable_tta, metadata, list_epoch, acc_train, acc_train_val, loss_train, loss_val, val_ds, test_ds, error, model_path="./modelos_treinados/Pytorch/", nome_projeto=None, enable_fold=False, grid_search=False):
    br_time = datetime.now(ZoneInfo("America/Sao_Paulo"))
    formatted_time = br_time.strftime("%d-%m-%y_%H-%M-%S")

    ## For the entire structure
    model_path = f'{model_path}{model_name}/'
    os.makedirs(model_path, exist_ok=True)

    if (enable_fold):
        ## For each fold
        model_path = f'{model_path}{metadata["fold"]}/'
        os.makedirs(model_path, exist_ok=True)

    metricas_datasetTeste = {}
    best_threshold_acc = -1

    if (not(grid_search)):
        best_threshold_acc = threshold_optimization(copy.deepcopy(obj_model), val_ds, metadata["batch_size"], error)
        tta_val(copy.deepcopy(obj_model), val_ds, metadata["batch_size"], error, best_threshold_acc).to_csv(f'{model_path}ablation_tta_valds_e{metadata["epoch"]}.csv', index=False)
        tta_val(copy.deepcopy(obj_model), test_ds, metadata["batch_size"], error, best_threshold_acc).to_csv(f'{model_path}ablation_tta_testds.csv', index=False)
        
        (tp_test, tn_test, fp_test, fn_test, acc_test, recall_test, prec_test, espec_test, f1_test, pAUC_test) = calc_statisticsV2_pytorch(obj_model, error, test_ds, enable_tta=enable_tta, visible=False)
        metricas_datasetTeste = {"tp_teste": tp_test,
             "tn_teste": tn_test,
             "fp_teste": fp_test,
             "fn_teste": fn_test,
             "acurácia_teste": acc_test,
             "recall_teste": recall_test,
             "precisão_teste": prec_test,
             "especificação_teste": espec_test,
             "f1-score_teste": f1_test,
             "pAUC_teste": pAUC_test}
    
    (tp_val, tn_val, fp_val, fn_val, acc_val, recall_val, prec_val, espec_val, f1_val, pAUC_val) = calc_statisticsV2_pytorch(obj_model, error, val_ds, enable_tta=enable_tta, visible=False)
    
    nome_pasta = ""

    data = {"config_modelo":
            metadata,
        "metricas":
            {"best_threshold_acc": best_threshold_acc,
             "acc_train": acc_train,
             "loss_train": loss_train,
             "acc_val": acc_train_val,
             "loss_val": loss_val},
        "metricas_datasetVal":
            {"tp_val": tp_val,
             "tn_val": tn_val,
             "fp_val": fp_val,
             "fn_val": fn_val,
             "acurácia_val": acc_val,
             "recall_val": recall_val,
             "precisão_val": prec_val,
             "especificação_val": espec_val,
             "f1-score_val": f1_val,
             "pAUC_teste": pAUC_val},
        "metricas_datasetTeste":
            metricas_datasetTeste,}

    ## Replace the placeholders with your own Google Sheet ID and Name
    if (enable_fold):
        model_name = f'{model_name}_f{metadata["fold"]}_e{metadata["epoch"]}_{formatted_time}'
    if (grid_search):
        model_name = f'{model_name}_e{metadata["epoch"]}'
    else:
        model_name = f'{model_name}_e{metadata["epoch"]}_{formatted_time}'
    
    saveCSV_RAY(model_name, data, google_sheet_id, google_sheet_name)

    plotGraphics(model_path, list_epoch, acc_train=acc_train, acc_val=acc_train_val, loss_train=loss_train, loss_val=loss_val)

    with open(f'{model_path}training_metadata.json', "w") as file:
        ## Stores essential model metadata
        json.dump(data, file, cls=NpEncoder)

    ## Saves the model's state_dict (the trained weights)
    torch.save(obj_model.state_dict(), f'{model_path}{model_name}.pth')
    
    ## If error, continues
    try:
        saved_nb = save_notebook_snapshot(model_path)
    except Exception as e:
        print(f"[warn] não consegui salvar o notebook: {e}")
        
    print(f'Saved on: {model_path}')

    del obj_model

In [ ]:
import time
import gspread

def getSheet(sheet_id, worksheet):
    while True:
        try:
            sheet = connnect_drive(sheet_id, worksheet)
            return sheet
        except:
            print("ERROR connecting with sheet, tying again...")
            time.sleep(105)

## Checks if a model with the exact given metadata already exists in the Google Sheet.
## Useful for resuming grid searches and avoiding duplicate training runs.
def checkExistsModel_CSV(metadata, sheet_id, worksheet=None):
    sheet = getSheet(sheet_id, worksheet)

    ## Selects the specified worksheet or defaults to the first one
    if (worksheet != None):
        sheet = sheet.worksheet(worksheet)
    else:
        sheet = sheet.sheet1

    ## Flattens nested dictionaries into a single dictionary
    dict_items = {}
    try:
      for i in metadata.keys():
          for key, value in metadata[i].items():
              if value == []:
                  metadata[i][key] = None
              if (type(value) == list):
                  metadata[i][key] = str(metadata[i][key])
          dict_items = (dict_items | metadata[i])
    except AttributeError:
        ## Fallback: Uses the dictionary directly if it is not nested
        dict_items = metadata
        pass

    ## Loads the Google Sheet into a pandas DataFrame
    df_carregado = gs_d.get_as_dataframe(sheet, index_col=1)

    lista_planilha = list((df_carregado.columns))
    lista_metadata = list((dict_items.keys()))

    ## Removes performance metrics from the comparison list, keeping only hyperparameters
    for element in ["nome", "Unnamed: 0", 'model_config', "tp_loss", "tn_loss", "fp_loss", "fn_loss", 'tp_teste', 'tn_teste', 'fp_teste', 'fn_teste', "train_loss", "test_loss", "acc_train", "acc_val", "loss_train", "loss_val", "acurácia_teste", "recall_teste", "precisão_teste", "especificação_teste", "f1-score_teste", "pAUC_teste"]:
        try:
          lista_planilha.remove(element)
        except:
          pass

    df = df_carregado.loc[:, lista_planilha]

    # Verify if second layer exists and if not adds the information to be verified afterwards
    dict_items.setdefault("camada_densa_2", 0)
    if "camada_densa_2" not in lista_metadata:
        lista_metadata.append("camada_densa_2")
        dict_items["camada_densa_2"] = 0
        df["camada_densa_2"] = df["camada_densa_2"].fillna(0)
    
    ## Filters the DataFrame iteratively to find rows matching the exact metadata
    for element in lista_planilha:
        try:
            df = df[df[element] == dict_items[element]]
            lista_metadata.remove(element)
        except KeyError:
            continue
    
    try:
      lista_metadata.remove('model_config')
    except KeyError:
      pass
        
    # Returns True if an exact match is found and all metadata keys were verified
    if ((len(df) >= 1) and (len(lista_metadata) == 0)):
        return True
    else:
        return False

In [ ]:
import os

## Add a HF_TOKEN to your .env file, representing a HugginFace token.
os.environ["HF_TOKEN"] = os.getenv('HF_TOKEN')
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

import timm
import torch.nn as nn
import torch.nn.functional as F

class CNNMedicalCustomTimm(nn.Module):
    def __init__(self, model_base, model_weights, layer=[512, 256], dropout_rate=0.4, discriminative_fine=True):
        super().__init__()
        self.device = device

        self.backbone = timm.create_model(
            "convnextv2_base.fcmae_ft_in22k_in1k",
            pretrained=True,
            num_classes=0,
        )

        data_config = timm.data.resolve_model_data_config(self.backbone)
        self.input_size = data_config["input_size"]      ## (3, 224, 224)
        self.mean = torch.tensor(data_config["mean"]).view(1, 3, 1, 1)
        self.std = torch.tensor(data_config["std"]).view(1, 3, 1, 1)

        output_features = self.backbone.num_features

        if (discriminative_fine == True):
            for p in self.backbone.parameters():
                p.requires_grad = False
        
            for p in self.backbone.stages[-1].parameters():
                p.requires_grad = True
        else:
            for p in self.backbone.parameters():
                p.requires_grad = True

        self.layers = nn.ModuleList()
        self.norm = nn.ModuleList()
        self.activations = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout_rate) if dropout_rate != 0 else None

        if (len(layer) != 0):
            for i, neurons in enumerate(layer):
                in_f = output_features if i == 0 else layer[i-1]
                self.layers.append(nn.Linear(in_f, neurons))
                self.norm.append(nn.BatchNorm1d(neurons))
                self.activations.append(nn.GELU())
            self.output = nn.Linear(layer[-1], 1)
        else:
            self.output = nn.Linear(output_features, 1)

    def preprocess_batch(self, x):
        x = x.to(self.device, non_blocking=True)

        if x.dtype != torch.float32:
            x = x.float()

        if x.max() > 1.0:
            x = x / 255.0

        _, h, w = self.input_size
        if x.shape[-2:] != (h, w):
            x = F.interpolate(x, size=(h, w), mode="bilinear", align_corners=False)

        mean = self.mean.to(x.device)
        std = self.std.to(x.device)
        x = (x - mean) / std
        return x


    def forward(self, x):
        x = self.preprocess_batch(x)

        x = self.backbone(x)

        ## Head
        for layer, norm, activation in zip(self.layers, self.norm, self.activations):
            x = activation(norm(layer(x)))
            if self.dropout is not None:
                x = self.dropout(x)

        output = self.output(x)
        return output

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, restore_best=True, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.mode = mode
        self.epoch = 0
        self.counter = 0
        self.best_score = None
        self.early_score = None
        self.early_stop = False
        self.best_weights = None

    def __call__(self, val_metric, epoch, model):
        score = val_metric if self.mode == "max" else -val_metric

        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            self.epoch = epoch
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

class BinaryFocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss) # prob de acertar a classe
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

import torch
import gc

def clean_memory():
    if 'model' in locals():
        del model
    if 'test_ds' in locals():
        del test_ds
    if 'error' in locals():
        del error

    gc.collect()

    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
def build_optimizer_and_scheduler(ml_model, opt, model_config, phase, head_lr, backbone_lr, phase_epochs=10):
    if phase == "head_only":
        params = list(ml_model.layers.parameters()) \
               + list(ml_model.norm.parameters()) \
               + list(ml_model.output.parameters())
        optimizer = opt(params, lr=head_lr, weight_decay=model_config["opt_weight_decay"])
    elif phase == "head_last_stage":
        optimizer = opt(
            [
                {
                    "params": ml_model.backbone.stages[-1].parameters(),
                    "lr": backbone_lr,
                },
                {
                    "params": list(ml_model.layers.parameters())
                            + list(ml_model.norm.parameters())
                            + list(ml_model.output.parameters()),
                    "lr": head_lr,
                },
            ],
            weight_decay=model_config["opt_weight_decay"]
        )
    elif phase == "full":
        optimizer = opt(
            [
                {"params": ml_model.backbone.parameters(), "lr": backbone_lr},
                {"params": list(ml_model.layers.parameters())
                            + list(ml_model.norm.parameters())
                            + list(ml_model.output.parameters()),
                 "lr": head_lr,},
            ],
            weight_decay=model_config["opt_weight_decay"]
        )
    else:
        raise ValueError(f"Fase inválida: {phase}")

    scheduler = None
    if model_config["enable_sch"]:
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=phase_epochs,
            eta_min=1e-6
        )

    return optimizer, scheduler

In [ ]:
from torch import nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import WeightedRandomSampler, ConcatDataset
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from peft import LoraConfig, get_peft_model
import copy
import numpy as np
import math

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
assert device == "cuda", "Running on CPU. A CUDA-enabled device is highly recommended. If you must run on CPU, comment out this line."

def train_search(nome_projeto, model_base, epoch, opt, learning_rate, batch_size, camada_densa=[], dropout=0.0, loss_alpha=0.75, loss_gamma=2.0, custom_threshold=0.5, model_path="./dados/Pytorch/ODJ/", grid_search=False, discriminative_fine=True):
        inicio = time.perf_counter()
        br_time = datetime.now(ZoneInfo("America/Sao_Paulo"))
        formatted_time = br_time.strftime("%d-%m-%y_%H-%M-%S")

        model_config = {"discriminative_fine": discriminative_fine, "enable_stagetrain": False, "warmup_epochs": 3, "enable_tta": True, "enable_es": False, "enable_sch": True, 
                        "enable_weightedsampler": False, "num_workers": 4, "loss_alpha": loss_alpha, "loss_gamma": loss_gamma, "opt_weight_decay": 0.05, "opt_lr_inicial": learning_rate, 
                        "opt_head_lr": 5e-4, "opt_backbone_lr": 1e-4}

        train_ds = DataLoader(full_dataset.get_dataset("train"), batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=model_config["num_workers"], prefetch_factor=2)
        val_ds = DataLoader(full_dataset.get_dataset("val"), batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=model_config["num_workers"], prefetch_factor=2)
        test_ds = DataLoader(full_dataset.get_dataset("test"), batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=model_config["num_workers"])

        ## Create Global Progress Bar
        train_batches_per_epoch = math.ceil(len(train_ds))
        val_batches_per_epoch = math.ceil(len(val_ds))
        
        TOTAL_STEPS = epoch * (train_batches_per_epoch + val_batches_per_epoch)
        pbar_global = tqdm(total=TOTAL_STEPS, desc="Progresso Global", position=0, leave=True, colour='green')

        clean_memory()

        ## ----------------------------------------------------
        ## Create model object
        
        match model_base:
            case "swin_v2_b":
                ml_model = NeuralNetworkCustom(model_base=swin_v2_b, model_weights=Swin_V2_B_Weights.DEFAULT, layer=camada_densa, dropout_rate=dropout, using_embeddings=using_embeddings, nome_projeto=nome_projeto)
            case "convnextv2_base.fcmae_ft_in22k_in1k":
                ml_model = CNNMedicalCustomTimm(model_base=convnext_base, model_weights=ConvNeXt_Base_Weights.DEFAULT, layer=camada_densa, dropout_rate=dropout, discriminative_fine=model_config["discriminative_fine"])
            case _:
                raise Exception("Modelo passado não é suportado.")

        ml_model = ml_model.to(device)
        ml_model = torch.compile(ml_model)
        
        basemodel_name = str(model_base)[str(model_base).find(" ") + 1:str(model_base).find(" at")] ## 'swin_v2_b'
        nome_pasta = f'{nome_projeto}'
        
        ## ----------------------------------------------------
        #3 Create optimizer, scaler and error function
        
        optimizer = opt(ml_model.parameters(), lr=learning_rate, weight_decay=model_config["opt_weight_decay"])
        opt_name = str(type(optimizer))[str(type(optimizer)).rindex('.') + 1:str(type(optimizer)).rindex("'")]

        scaler = torch.amp.GradScaler('cuda')
        
        error = BinaryFocalLossWithLogits(alpha=model_config["loss_alpha"], gamma=model_config["loss_gamma"])

        ## ----------------------------------------------------
        ## Preparing Metadata
        
        metadata = {"modelo_base": model_base, "epoch": epoch, "custom_threshold": custom_threshold, "learning_rate": learning_rate, "weight_decay": model_config["opt_weight_decay"], 
                    "optimizer": opt_name, "batch_size": batch_size, "dropout": dropout, "model_config": str(model_config)}
        metadata_new = metadata.copy()

        acc = 1
        for element in camada_densa:
            metadata_new["camada_densa_" + str(acc)] = element
            acc += 1
        metadata = metadata_new
        del metadata_new

        metadata.setdefault("camada_densa_2", 0)

        ## Purely an existence check (Google Sheets acts as the single source of truth for saved configurations)
        ## Replace the placeholders with your own Google Sheet Name
        if (grid_search):
            try:
              if (checkExistsModel_CSV(metadata=metadata, sheet_id=google_sheet_id, worksheet=google_sheet_name)):
                  print("Modelo já treinado, continuando...")
                  return 0
            except gspread.exceptions.APIError:
              print("Quota Limited, waiting...")
              time.sleep(105)
              if (checkExistsModel_CSV(metadata=metadata, sheet_id=google_sheet_id, worksheet=google_sheet_name)):
                print("Modelo já treinado, continuando...")
                return 0

        ## ----------------------------------------------------

        list_epoch = []
        acc_train = []
        loss_train = []
        acc_val = []
        loss_val = []

        if (model_config["enable_es"] == True) and (grid_search == False):
            es = EarlyStopping(patience=5, mode="max")
            es_exportado = False

        if (model_config["enable_sch"] == True):
            scheduler = CosineAnnealingLR(optimizer, T_max=epoch, eta_min=1e-6)

        for iter in range(epoch):
            ## Train Phase 
            ml_model.train()
            dataloader = train_ds
            
            if (model_config["enable_sch"] == True):
                current_lr = scheduler.get_last_lr()[0]
            else:
                current_lr = learning_rate

            if (model_config["enable_stagetrain"] == True) and (iter == model_config["warmup_epochs"]):
                print("Going to the next stage")
                phase = "head_last_stage"
                ml_model.set_train_phase(phase)
                optimizer, scheduler = build_optimizer_and_scheduler(ml_model, opt, model_config, phase, head_lr=model_config["opt_head_lr"], backbone_lr=model_config["opt_backbone_lr"], phase_epochs=(epoch-(iter+1)))

            running_acc = 0
            running_loss = 0

            total_samples = 0

            pbar = tqdm(dataloader, desc=f"Train Epoch {iter+1}/{epoch}",  postfix={"loss": 0.0, "accuracy": 0.0, "LR": current_lr}, disable=False)
            pbar.ncols = 0 ## Disable bar but iteration remains

            for i, (images, labels) in enumerate(pbar):
                images = images.to(device)
                labels = labels.to(device).float().view(-1, 1)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast('cuda'):
                    outputs = ml_model(images)
                    loss = error(outputs, labels)

                batch_acc = 0
                batch_loss = 0

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                train_batch_size = images.size(0)
                total_samples += train_batch_size
                running_loss += loss.item() * train_batch_size

                with torch.no_grad():
                    probs = torch.sigmoid(outputs)
                    preds = (probs > custom_threshold).float()
                    batch_acc = (preds == labels).float().sum().item()
                    running_acc += batch_acc

                current_loss = running_loss / total_samples
                current_acc = running_acc / total_samples
                pbar.set_postfix({"loss": f"{current_loss:.4f}", "accuracy": f"{current_acc:.4f}", "LR": current_lr})
                pbar_global.update(1)

            if (model_config["enable_sch"] == True):
                scheduler.step()

            ## End of epoch
            train_acc = running_acc / total_samples
            train_loss = running_loss / total_samples

            acc_train.append(train_acc)
            loss_train.append(train_loss)
            list_epoch.append(iter)

            ## Validation Phase 
            ml_model.eval()
            dataloader = val_ds

            running_acc = 0
            running_loss = 0

            total_samples = 0

            pbar = tqdm(dataloader, desc=f"Val Epoch {iter+1}/{epoch}",  postfix={"loss": 0.0, "accuracy": 0.0}, disable=False)
            pbar.ncols = 0

            with torch.no_grad():
                for i, (images, labels) in enumerate(pbar):
                    images = images.to(device)
                    labels = labels.to(device).float().view(-1, 1)

                    with torch.amp.autocast('cuda'):
                        outputs = ml_model(images)
                        loss = error(outputs, labels)

                    batch_acc = 0
                    batch_loss = 0

                    val_batch_size = images.size(0)
                    total_samples += val_batch_size
                    running_loss += loss.item() * val_batch_size

                    probs = torch.sigmoid(outputs)
                    preds = (probs > custom_threshold).float()
                    batch_acc = (preds == labels).float().sum().item()
                    running_acc += batch_acc

                    current_loss = running_loss / total_samples
                    current_acc = running_acc / total_samples
                    pbar.set_postfix({"loss": f"{current_loss:.4f}", "accuracy": f"{current_acc:.4f}"})
                    pbar_global.update(1)

                val_loss = running_loss / total_samples
                val_acc = running_acc / total_samples

                acc_val.append(val_acc)
                loss_val.append(val_loss)

            ## Early Stopping for training
            if ((model_config["enable_es"] == True) and (grid_search == False) and (not es_exportado)):
                es(val_acc, iter, ml_model)
                if es.early_stop:
                    print(F"ES: Carregando o modelo da epoch {es.epoch}")
                    es_model = copy.deepcopy(ml_model)                        
                    es_model.load_state_dict(es.best_weights)  ## restore best
                    
                    fim = time.perf_counter()
                    info_loss = {"train_loss": [train_loss], "test_loss": [val_loss], "tempo": str(timedelta(seconds=(fim - inicio)))}
                    model_nome = f'{basemodel_name}_{formatted_time}_e{epoch}_o{metadata["optimizer"]}_lr{np.round(metadata["learning_rate"], 5)}_b{metadata["batch_size"]}_{dataset_nome}_{nome_projeto}'
                    metadata["epoch"] = es.epoch+1
                    saveModelV2(es_model, model_nome, enable_tta=model_config["enable_tta"], metadata=(metadata | info_loss), list_epoch=list_epoch, acc_train=acc_train, acc_train_val=acc_val, loss_train=loss_train, loss_val=loss_val, val_ds=val_ds, test_ds=test_ds, error=error, model_path=model_path, nome_projeto=nome_projeto)
                    ml_model.train()
                    es_exportado = True
    
        fim = time.perf_counter()
        info_loss = {"train_loss": [train_loss], "test_loss": [val_loss], "tempo": str(timedelta(seconds=(fim - inicio)))}
        model_nome = f'{basemodel_name}_{formatted_time}_e{epoch}_o{metadata["optimizer"]}_lr{np.round(metadata["learning_rate"], 5)}_b{metadata["batch_size"]}_{dataset_nome}_{nome_projeto}'
        metadata["epoch"] = epoch
        saveModelV2(ml_model, model_nome, enable_tta=model_config["enable_tta"], metadata=(metadata | info_loss), list_epoch=list_epoch, acc_train=acc_train, acc_train_val=acc_val, loss_train=loss_train, loss_val=loss_val, val_ds=val_ds, test_ds=test_ds, error=error, model_path=model_path, nome_projeto=nome_projeto, grid_search=grid_search)

## Model Training and Hyperparameter Grid Search

In [ ]:
## Main Model + Ablation

import time
from datetime import timedelta
from pathlib import Path

## ----------------------------------------------------

## Model settings
learning_rate = 0.001
batch_size = 32
camada_densa = [512, 128]
dropout = 0
custom_threshold = 0.5

## Training settings
google_sheet_name = "[Insert your Google Sheet Name]"
model_path = f'./dados/Pytorch/ODJ/Revisado/Resultados_Finais'

## ----------------------------------------------------

os.makedirs(model_path, exist_ok=True)

inicio = time.perf_counter()
ml_model = train_search(nome_projeto="full_model", model_base="convnextv2_base.fcmae_ft_in22k_in1k", epoch=30, opt=torch.optim.AdamW, learning_rate=learning_rate, batch_size=batch_size, camada_densa=camada_densa, dropout=dropout, model_path=model_path, grid_search=False)
ml_model = train_search(nome_projeto="without_mlp", model_base="convnextv2_base.fcmae_ft_in22k_in1k", epoch=30, opt=torch.optim.AdamW, learning_rate=learning_rate, batch_size=batch_size, camada_densa=[], dropout=dropout, model_path=model_path, grid_search=False)
ml_model = train_search(nome_projeto="full_training", discriminative_fine=False, model_base="convnextv2_base.fcmae_ft_in22k_in1k", epoch=30, opt=torch.optim.AdamW, learning_rate=learning_rate, batch_size=batch_size, camada_densa=camada_densa, dropout=dropout, model_path=model_path, grid_search=False)

## Change to dataset without data augmentation
datasetDirectory = "./datasets/discos_11732_semAG/"
ssd_datasetDirectory = "/home/jovyan/SSD/" + datasetDirectory[11:]
dataset_nome = datasetDirectory[datasetDirectory.rindex("/", 0, (len(datasetDirectory) - 1)) + 1:len(datasetDirectory) - 1]
full_dataset = DatasetGlaucomaTVT(
   img_dir=ssd_datasetDirectory,
   transform=ToTensor()
)

ml_model = train_search(nome_projeto="without_aug", model_base="convnextv2_base.fcmae_ft_in22k_in1k", epoch=30, opt=torch.optim.AdamW, learning_rate=learning_rate, batch_size=batch_size, camada_densa=camada_densa, dropout=dropout, model_path=model_path, grid_search=False)
ml_model = train_search(nome_projeto="baseline", discriminative_fine=False, model_base="convnextv2_base.fcmae_ft_in22k_in1k", epoch=30, opt=torch.optim.AdamW, learning_rate=learning_rate, batch_size=batch_size, camada_densa=[], dropout=dropout, model_path=model_path, grid_search=False)
fim = time.perf_counter()

tempo_formatado = str(timedelta(seconds=(fim - inicio)))
print(f"Time: {tempo_formatado}")

In [ ]:
## GridSearch (FULL SEARCH)

import time
from datetime import timedelta
from sklearn.model_selection import ParameterGrid
from IPython.display import clear_output

config = {
    "model_base": ["convnextv2_base.fcmae_ft_in22k_in1k"],
    "layers": [[256], [512], [512, 128], [512, 256]],
    "optimizer": [torch.optim.AdamW],
    "lr": [0.001, 0.0005],
    "batch_size": [32],
    "dropout": [0, 0.2, 0.4] 
}

machine_id = 1   # 0 for Google Colab, 1 for Local

qtdEpoch = 15
custom_threshold = 0.5
google_sheet_name = "[Insert your Google Sheet Name]"

grid = list(ParameterGrid(config))
num_possibilidades = len(grid)
print(num_possibilidades)

mid = num_possibilidades // 2
if machine_id == 0:
    subset = grid[:mid]
else:
    subset = list(reversed(grid[mid:]))

model_path = f'./dados/Pytorch/ODJ/GridSearch/1/'
os.makedirs(model_path, exist_ok=True)

num_possibilidades_subset = len(subset)
pbar_gridsearch = tqdm(total=num_possibilidades_subset, desc="GridSearch", position=0, leave=True, colour='blue')
for i, dict in enumerate(subset):
    ml_model = train_search(nome_projeto="full_model", model_base=dict["model_base"], epoch=qtdEpoch, opt=dict["optimizer"], learning_rate=dict["lr"], batch_size=dict["batch_size"], camada_densa=dict["layers"], dropout=dict["dropout"], model_path=model_path, grid_search=True)
    pbar_gridsearch.update(1)

In [ ]:
## Confusion Matrix

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
import numpy as np

tp = 282
tn = 792
fp = 42
fn = 57

cm = np.array([
    [tn, fp],
    [fn, tp]
])

plt.figure(figsize=(7, 5))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                 xticklabels=['Negative', 'Positive'], 
                 yticklabels=['Negative', 'Positive'])


plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)

plt.yticks(rotation=0) 

plt.tight_layout()
plt.show()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')